# Preprocessing

In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from pathlib import Path
from src.aggregate import aggregate

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")

In [2]:
bureau = pd.read_csv(RAW / "bureau.csv").merge(pd.read_pickle(INTERIM / "bureau_balance.pkl"), on="SK_ID_BUREAU", how="left")
top = ["Consumer credit", "Credit card", "Car loan", "Mortgage", "Microloan"]
bureau["CREDIT_TYPE"] = bureau["CREDIT_TYPE"].where(bureau["CREDIT_TYPE"].isin(top), "Other")
bureau = bureau.sort_values(["SK_ID_CURR", "DAYS_CREDIT"], ascending=[True, False])
bureau["days_credit_diff"] = bureau.groupby("SK_ID_CURR")["DAYS_CREDIT"].diff()
bureau["credit_ongoing"] = (bureau["DAYS_CREDIT_ENDDATE"] > 0).astype("int8")
bureau["unusual_currency"] = (bureau["CREDIT_CURRENCY"] != "currency 1").astype("int8")
bureau.shape

(1716428, 99)

# Feature Engineering

## Iteration 1

In [3]:
b = aggregate(bureau, "SK_ID_CURR", "bureau", cat_cols=["CREDIT_ACTIVE", "CREDIT_TYPE"], ignore=["SK_ID_BUREAU"])
b["bureau_debt_credit"] = b["bureau_AMT_CREDIT_SUM_DEBT_sum"] / b["bureau_AMT_CREDIT_SUM_sum"]
b["bureau_overdue_debt"] = b["bureau_AMT_CREDIT_SUM_OVERDUE_sum"] / b["bureau_AMT_CREDIT_SUM_DEBT_sum"]
b["bureau_annuity_credit"] = b["bureau_AMT_ANNUITY_sum"] / b["bureau_AMT_CREDIT_SUM_sum"]
b = b.replace([np.inf, -np.inf], np.nan)

In [4]:
f = pd.DataFrame({"SK_ID_CURR": bureau["SK_ID_CURR"].to_numpy()})
f["distress"] = bureau["CREDIT_ACTIVE"].isin(["Bad debt", "Sold"]).to_numpy("int8")
f["open"] = bureau["DAYS_ENDDATE_FACT"].isna().to_numpy("int8")
f["overdue"] = (bureau["AMT_CREDIT_SUM_OVERDUE"].fillna(0) > 0).to_numpy("int8")
g = f.groupby("SK_ID_CURR")
b = b.join(pd.DataFrame({
    "bureau_any_bad_debt": g["distress"].max(),
    "bureau_bad_debt_share": g["distress"].mean(),
    "bureau_share_open": g["open"].mean(),
    "bureau_any_overdue": g["overdue"].max(),
}))
b.shape

(305811, 493)

In [5]:
b["bureau_n_loan_types"] = bureau.groupby("SK_ID_CURR")["CREDIT_TYPE"].nunique()
b["bureau_loans_per_type"] = b["bureau_count"] / b["bureau_n_loan_types"]

parts = []
for d in [365, 730, 1095, 1825]:
    w = bureau[bureau["DAYS_CREDIT"] >= -d].groupby("SK_ID_CURR")
    parts += [w.size().rename(f"bureau_n_loans_{d}"),
              w["AMT_CREDIT_SUM_DEBT"].sum().rename(f"bureau_debt_{d}"),
              w["AMT_CREDIT_SUM_OVERDUE"].sum().rename(f"bureau_overdue_{d}"),
              w["credit_ongoing"].mean().rename(f"bureau_active_share_{d}"),
              w["days_credit_diff"].mean().rename(f"bureau_days_between_{d}"),
              w["CNT_CREDIT_PROLONG"].sum().rename(f"bureau_prolong_{d}")]
b = b.join(pd.concat(parts, axis=1))
b.shape

(305811, 519)

## Iteration 2

In [6]:
for state, tag in [("Active", "active"), ("Closed", "closed")]:
    sub = bureau[bureau["CREDIT_ACTIVE"] == state]
    a = aggregate(sub, "SK_ID_CURR", f"bureau_{tag}", cat_cols=["CREDIT_TYPE"], ignore=["SK_ID_BUREAU"])
    b = b.join(a)
b = b.replace([np.inf, -np.inf], np.nan)
b.shape

(305811, 1486)

# Save

In [7]:
b.reset_index().to_pickle(INTERIM / "bureau.pkl")
b.shape

(305811, 1486)